In [9]:
import pandas as pd
from rapidfuzz import process, fuzz

In [2]:
# Exploración de los dataset
df_global = pd.read_csv("../data/raw/global_football_results.csv")
df_global.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [3]:
df_wc = pd.read_csv("../data/raw/world_cup_last_50_years.csv")
df_wc.head()

,year,date,stage,home_team,away_team,home_goals,away_goals,winner,match_id
0,1974,1974-06-10,Quarter-finals,South Korea,Sweden,0,0,Draw,1
1,1974,1974-06-10,Quarter-finals,Ghana,Switzerland,2,0,Ghana,2
2,1974,1974-06-10,Group Stage,Sweden,Saudi Arabia,2,0,Sweden,3
3,1974,1974-06-10,Group Stage,Cameroon,Uruguay,2,1,Cameroon,4
4,1974,1974-06-10,Group Stage,Nigeria,Japan,2,1,Nigeria,5


In [4]:
# Primer análisis
df_global.info()
df_wc.info()

<class 'pandas.DataFrame'>
RangeIndex: 47399 entries, 0 to 47398
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        47399 non-null  str  
 1   home_team   47399 non-null  str  
 2   away_team   47399 non-null  str  
 3   home_score  47399 non-null  int64
 4   away_score  47399 non-null  int64
 5   tournament  47399 non-null  str  
 6   city        47399 non-null  str  
 7   country     47399 non-null  str  
 8   neutral     47399 non-null  bool 
dtypes: bool(1), int64(2), str(6)
memory usage: 2.9 MB
<class 'pandas.DataFrame'>
RangeIndex: 832 entries, 0 to 831
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   year        832 non-null    int64
 1   date        832 non-null    str  
 2   stage       832 non-null    str  
 3   home_team   832 non-null    str  
 4   away_team   832 non-null    str  
 5   home_goals  832 non-null    int64
 6   away_goal

In [10]:
df_global["home_team"] = df_global["home_team"].str.strip()
df_global["away_team"] = df_global["away_team"].str.strip()
df_wc["home_team"] = df_global["home_team"].str.strip()
df_wc["away_team"] = df_global["away_team"].str.strip()

In [11]:
# Nombres consistentes
all_teams = sorted(
    set(df_global["home_team"]).union(set(df_global["away_team"]))
)

In [12]:
def normalize_team(name, choices, threshold=90):
    name = str(name).strip()

    match, score, _ = process.extractOne(
        name,
        choices,
        scorer=fuzz.WRatio
    )

    if score >= threshold:
        return match
    else:
        return name

In [13]:
df_global["home_team"] = df_global["home_team"].apply(
    lambda x: normalize_team(x, all_teams)
)

df_global["away_team"] = df_global["away_team"].apply(
    lambda x: normalize_team(x, all_teams)
)

In [14]:
# Crear variable objetivo
def get_result(row, home_col, away_col):
    if row[home_col] > row[away_col]:
        return 1
    elif row[home_col] < row[away_col]:
        return -1
    else:
        return 0

df_global["result"] = df_global.apply(
    get_result, 
    axis=1, 
    args=("home_score", "away_score")
)

df_wc["result"] = df_wc.apply(
    get_result,
    axis=1,
    args=("home_goals", "away_goals")
)

In [15]:
# Crear memoria de equipo
team_stats = {}
teams = pd.concat([df_global["home_team"], df_global["away_team"]]).unique()
for team in teams:
    team_stats[team] = {"games": 0, "wins": 0}

In [16]:
# Llenado
for _, row in df_global.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    result = row["result"]
    # partidos
    team_stats[home]["games"] +=1
    team_stats[away]["games"] +=1
    # victorias
    if result == 1:
        team_stats[home]["wins"] +=1
    elif result == -1:
        team_stats[away]["wins"] += 1

In [17]:
# Fuerza 
def win_rate(team):
    stats = team_stats[team]
    if stats["games"] == 0:
        return 0.5
    return stats["wins"] / stats["games"]

In [24]:
win_rate("Mexico")

0.5107692307692308

In [25]:
win_rate("Argentina")

0.5501432664756447

In [26]:
win_rate("Ghana")

0.4709480122324159

In [27]:
df_global = df_global.copy()
df_global["date"] = pd.to_datetime(df_global["date"])
df_global = df_global.sort_values("date").reset_index(drop=True)

In [28]:
# (Elo) medir fuerza relativa
elo = {}
teams = pd.concat([
    df_global["home_team"],
    df_global["away_team"]
]).unique()
# misma fuerza para todos al inicio
for team in teams:
    elo[team] = 1500

In [29]:
# Convertir diferencia de fuerza en probabilidad
def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

In [30]:
# Ajusta fuerrza
def update_elo(home, away, home_score, away_score, k=20):
    home_elo = elo.get(home, 1500)
    away_elo = elo.get(away, 1500)

    exp_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))

    if home_score > away_score:
        result_home = 1
    elif home_score < away_score:
        result_home = 0
    else:
        result_home = 0.5

    elo[home] = home_elo + k * (result_home - exp_home)
    elo[away] = away_elo + k * ((1 - result_home) - (1 - exp_home))

In [31]:
for _, row in df_global.iterrows():
    update_elo(
        row["home_team"],
        row["away_team"],
        row["home_score"],
        row["away_score"]
    )

In [32]:
# Convertir Elo en vaiables
df_global["home_elo"] = df_global["home_team"].map(elo)
df_global["away_elo"] = df_global["away_team"].map(elo)

df_global["elo_diff"] = df_global["home_elo"] - df_global["away_elo"]

In [33]:
rows = []

In [34]:
# Construcción de dataset y actualización del modelo
elo = {}
for _, row in df_global.iterrows():
    home = row["home_team"]
    away = row["away_team"]

    home_elo = elo.get(home, 1500)
    away_elo = elo.get(away, 1500)

    rows.append({
        "home_team": home,
        "away_team": away,
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_diff": home_elo - away_elo,
        "result": row["result"]
    })

    update_elo(home, away, row["home_score"], row["away_score"])

In [35]:
df_ml = pd.DataFrame(rows)

In [36]:
df_ml.head()

,home_team,away_team,home_elo,away_elo,elo_diff,result
0,Scotland,England,1500.000000,1500.000000,0.000000,0
1,England,Scotland,1500.000000,1500.000000,0.000000,1
2,Scotland,England,1490.000000,1510.000000,-20.000000,1
3,England,Scotland,1499.424989,1500.575011,-1.150023,0
4,Scotland,England,1500.541911,1499.458089,1.083822,1


In [37]:
df_ml["elo_diff"].describe()

count    47399.000000
mean         8.421145
std        157.520621
min       -862.339139
25%        -87.566514
50%          8.429364
75%        106.007442
max        821.628349
Name: elo_diff, dtype: float64

In [38]:
df_ml = pd.DataFrame(rows)

In [39]:
df_wc["home_elo"] = df_wc["home_team"].map(elo)
df_wc["away_elo"] = df_wc["away_team"].map(elo)
df_wc["elo_diff"] = df_wc["home_elo"] - df_wc["away_elo"]

In [40]:
df_ml.to_csv("../data/processed/elo_dataset.csv", index=False)
df_wc.to_csv("../data/processed/wc_features.csv", index=False)